# 🛍️ ShopUNow Agentic AI Assistant — Demo Notebook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suranjika/ShopUNow-AI-Assistant/blob/main/notebooks/ShopUNow_Demo.ipynb)

---

**Author:** Suranjika Sahu — Data & Analytics Leader | AI/GenAI | Jakarta  
**Program:** Analytics Vidhya — Agentic AI Pioneer Program (2025)  
**GitHub:** [ShopUNow-AI-Assistant](https://github.com/suranjika/ShopUNow-AI-Assistant)

---

## 🎯 What This Notebook Covers

This notebook walks through the complete **ShopUNow Agentic AI Assistant** end-to-end:

| Step | What we build |
|---|---|
| 1️⃣ | Install dependencies |
| 2️⃣ | Set up Gemini API key securely |
| 3️⃣ | Build the department knowledge base |
| 4️⃣ | Create FAISS vector store with Gemini embeddings |
| 5️⃣ | Define the LangGraph agent nodes |
| 6️⃣ | Build and compile the StateGraph workflow |
| 7️⃣ | Visualise the agent flow |
| 8️⃣ | Run test queries and inspect outputs |
| 9️⃣ | Launch the interactive assistant |

---

## 🏗️ Architecture Overview

```
User Query
    │
    ▼
┌─────────────┐
│ Router Node │  ──►  HR / Finance / Billing / Shipping / Unknown
└─────────────┘
    │
    ▼
┌────────────────┐
│ Sentiment Node │  ──►  positive / neutral / negative
└────────────────┘
    │
    ▼
┌──────────────────┐
│ Escalation Node  │  ──►  escalate? yes/no
└──────────────────┘
    │
    ├── escalate=True  ──► Human Node  ──► END
    └── escalate=False ──► RAG Node    ──► END
```


---
## 📦 Step 1 — Install Required Packages

We install LangChain, LangGraph, Gemini (Google Generative AI), and FAISS for vector search.

> ⏱️ This may take 1–2 minutes on first run.

In [ ]:
!pip install -q \
    langchain \
    langchain-google-genai \
    langgraph \
    google-generativeai \
    faiss-cpu \
    python-dotenv

print('✅ All packages installed successfully!')

---
## 🔑 Step 2 — Set Up Your Gemini API Key (Securely)

**Never hardcode API keys in notebooks.** Use one of these two methods:

**Option A — Google Colab (recommended):**  
Use Colab Secrets: click the 🔑 key icon in the left sidebar → add `GOOGLE_API_KEY`

**Option B — Local environment:**  
Create a `.env` file in the project root with `GOOGLE_API_KEY=your_key_here`

> Get a free key at: https://aistudio.google.com/app/apikey

In [ ]:
import os

# ── Option A: Google Colab secrets ────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab secrets")

# ── Option B: Local .env file ──────────────────────────────────────────────────
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if os.environ.get("GOOGLE_API_KEY"):
        print("✅ API key loaded from .env file")
    else:
        raise ValueError(
            "❌ GOOGLE_API_KEY not found.\n"
            "Add it to Colab secrets or create a .env file with:\n"
            "GOOGLE_API_KEY=your_key_here"
        )

---
## 📚 Step 3 — Import Core Libraries

In [ ]:
from typing import TypedDict, List

# LangChain + Gemini
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.schema import Document, HumanMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS

# LangGraph
from langgraph.graph import StateGraph, END

print("✅ All libraries imported successfully!")

---
## 🗂️ Step 4 — Build the Department Knowledge Base

We define 48 FAQ entries across 4 departments:

| Department | Audience | # FAQs |
|---|---|---|
| HR | Internal (employees) | 12 |
| Finance | Internal (employees) | 12 |
| Billing | External (customers) | 12 |
| Shipping | External (customers) | 12 |

Each entry has: `question`, `answer`, `department`, `audience`.

In [ ]:
DATASETS = {
    "HR": [
        {"q": "How do I apply for paid time off (PTO)?", "a": "Submit a PTO request in Workday > Time Off. Manager approval required. Submit at least 3 business days in advance.", "department": "HR", "audience": "internal"},
        {"q": "What are ShopUNow's core working hours?", "a": "Core hours are 10:00–16:00 local time. Flexible start/end as agreed with your manager.", "department": "HR", "audience": "internal"},
        {"q": "Does ShopUNow offer parental leave?", "a": "Yes. 16 weeks paid primary caregiver leave and 6 weeks paid secondary caregiver leave.", "department": "HR", "audience": "internal"},
        {"q": "How do I update my legal name?", "a": "Open an HR ticket with legal documentation. HR will update payroll and directory within 5 business days.", "department": "HR", "audience": "internal"},
        {"q": "How can I access the employee handbook?", "a": "The handbook is in the HR Portal > Documents. Latest version is always pinned.", "department": "HR", "audience": "internal"},
        {"q": "Whom do I contact for workplace conflict mediation?", "a": "Create a confidential HR case in the HR Portal. An HRBP will reach out within 2 business days.", "department": "HR", "audience": "internal"},
        {"q": "Are there public holidays by region?", "a": "Yes. Holidays follow local calendars and are listed in the HR Portal > Regional Holidays.", "department": "HR", "audience": "internal"},
        {"q": "How do performance reviews work?", "a": "Biannual reviews via Workday: self-review, peer/manager feedback, calibration, final rating.", "department": "HR", "audience": "internal"},
        {"q": "Can I work remotely?", "a": "Remote/hybrid depends on role and manager approval. See Remote Work Policy in HR Portal.", "department": "HR", "audience": "internal"},
        {"q": "Where do I report harassment?", "a": "Use the confidential Ethics & Compliance form or contact HRBP immediately.", "department": "HR", "audience": "internal"},
        {"q": "How do I enroll in benefits?", "a": "Enroll during onboarding or open enrollment via Benefits Center in Workday.", "department": "HR", "audience": "internal"},
        {"q": "What training is mandatory?", "a": "Security, Code of Conduct, and Anti-harassment trainings annually via LMS.", "department": "HR", "audience": "internal"},
    ],
    "Finance": [
        {"q": "What is the expense reimbursement policy?", "a": "Submit expenses within 30 days via Concur. Receipts required for items > $25. Travel must be pre-approved.", "department": "Finance", "audience": "internal"},
        {"q": "How long do reimbursements take?", "a": "Approved claims are paid in the next weekly AP run (typically 5–7 business days).", "department": "Finance", "audience": "internal"},
        {"q": "Can I buy software with my card?", "a": "All software requires vendor security review and Finance approval before purchase.", "department": "Finance", "audience": "internal"},
        {"q": "What's the fiscal year?", "a": "ShopUNow fiscal year runs Jan 1 – Dec 31.", "department": "Finance", "audience": "internal"},
        {"q": "How are per diems handled?", "a": "Per diems follow government/region tables; claim via Concur with travel dates and destination.", "department": "Finance", "audience": "internal"},
        {"q": "Where do I find cost centers?", "a": "Cost center list is in the Finance Wiki. Ask your manager for your team's code.", "department": "Finance", "audience": "internal"},
        {"q": "How do vendor invoices get paid?", "a": "Vendors email invoices to ap@shopunow.com with PO. Net-30 unless negotiated.", "department": "Finance", "audience": "internal"},
        {"q": "Who approves capital expenses?", "a": "Department head and Finance Controller must approve capex > $5,000.", "department": "Finance", "audience": "internal"},
        {"q": "What's the company travel card policy?", "a": "Issued to frequent travelers. Card use limited to travel-related expenses.", "department": "Finance", "audience": "internal"},
        {"q": "Exchange rate for international claims?", "a": "Use Concur's daily rate on the expense date unless a receipt shows the charged currency.", "department": "Finance", "audience": "internal"},
        {"q": "How do I request a PO?", "a": "Submit a PR in ProcureNow with vendor, quote, and cost center. PO created after approvals.", "department": "Finance", "audience": "internal"},
        {"q": "Who to contact for payroll issues?", "a": "Open a Payroll ticket in the Finance Portal. Response in 1–2 business days.", "department": "Finance", "audience": "internal"},
    ],
    "Billing": [
        {"q": "Why was my card declined?", "a": "Common reasons include insufficient funds or bank security checks. Try another card or contact your bank.", "department": "Billing", "audience": "external"},
        {"q": "How do I download my invoice?", "a": "Go to Account > Orders, select the order, then click Download Invoice.", "department": "Billing", "audience": "external"},
        {"q": "Can I change the billing address after purchase?", "a": "Yes, within 30 minutes of checkout from Order Details > Edit Billing Address.", "department": "Billing", "audience": "external"},
        {"q": "Do you accept BNPL?", "a": "Yes, we support ShopPay Installments and Klarna where available.", "department": "Billing", "audience": "external"},
        {"q": "Why do I see two charges?", "a": "You may see a temporary authorization hold and the final charge. Holds drop off in 3–5 business days.", "department": "Billing", "audience": "external"},
        {"q": "How do I apply a promo code?", "a": "Enter the code at checkout in the Promo field before payment.", "department": "Billing", "audience": "external"},
        {"q": "Can I split payment methods?", "a": "Yes, you can split between one card and a gift card/store credit.", "department": "Billing", "audience": "external"},
        {"q": "Refund timeline?", "a": "Once approved, refunds post to your bank in 5–10 business days.", "department": "Billing", "audience": "external"},
        {"q": "Currency support?", "a": "We charge in your local currency where supported; otherwise in USD with your bank's conversion.", "department": "Billing", "audience": "external"},
        {"q": "VAT invoice available?", "a": "Yes. Add your tax ID at checkout; invoice will include VAT details.", "department": "Billing", "audience": "external"},
        {"q": "How to update saved cards?", "a": "Account > Payment Methods to add/remove cards securely.", "department": "Billing", "audience": "external"},
        {"q": "My promo code isn't working.", "a": "Check expiration, minimum spend, or category exclusions; contact support if issues persist.", "department": "Billing", "audience": "external"},
    ],
    "Shipping": [
        {"q": "When will my order ship?", "a": "Orders ship within 1–2 business days. You'll receive a tracking email once dispatched.", "department": "Shipping", "audience": "external"},
        {"q": "How do I track my package?", "a": "Use the tracking link in your email or go to Account > Orders and click Track.", "department": "Shipping", "audience": "external"},
        {"q": "Do you offer expedited shipping?", "a": "Yes: Standard, Expedited (2-day), and Priority (next business day) at checkout.", "department": "Shipping", "audience": "external"},
        {"q": "What if my package is late?", "a": "If tracking hasn't updated for 3 business days, contact support to open a carrier trace.", "department": "Shipping", "audience": "external"},
        {"q": "International shipping fees?", "a": "Shown at checkout. Customs/duties may apply and are displayed when available.", "department": "Shipping", "audience": "external"},
        {"q": "Can I change the address after ordering?", "a": "Edits are possible within 30 minutes from Order Details > Edit Shipping Address.", "department": "Shipping", "audience": "external"},
        {"q": "My tracking shows delivered but I didn't receive it.", "a": "Check neighbors and safe places. If not found in 24 hours, contact support for a lost-package claim.", "department": "Shipping", "audience": "external"},
        {"q": "Do you ship to PO boxes?", "a": "Yes for Standard shipping, not for Priority.", "department": "Shipping", "audience": "external"},
        {"q": "What is the return window?", "a": "30 days from delivery for most items. Start a return from Account > Returns.", "department": "Shipping", "audience": "external"},
        {"q": "How are fragile items packed?", "a": "We use protective materials and 'Fragile' labels; report damage with photos within 7 days.", "department": "Shipping", "audience": "external"},
        {"q": "Can I hold a shipment?", "a": "After dispatch, some carriers support Hold at Location; use your tracking link to request.", "department": "Shipping", "audience": "external"},
        {"q": "What are free-shipping thresholds?", "a": "Free Standard shipping on domestic orders over $50 (after discounts).", "department": "Shipping", "audience": "external"},
    ],
}

total = sum(len(v) for v in DATASETS.values())
print(f"✅ Knowledge base loaded: {total} FAQ entries across {len(DATASETS)} departments")
for dept, entries in DATASETS.items():
    print(f"   {dept}: {len(entries)} entries")

---
## 📊 Step 5 — Build FAISS Vector Store

We:
1. Convert FAQ entries into LangChain `Document` objects with department metadata
2. Split documents into chunks using `RecursiveCharacterTextSplitter`
3. Generate embeddings using Gemini `embedding-001`
4. Index them in FAISS for fast semantic search

> ⏱️ This step calls the Gemini API and may take 20–30 seconds.

In [ ]:
# Initialise Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Build Document objects with metadata
docs = []
for dept, entries in DATASETS.items():
    for entry in entries:
        docs.append(Document(
            page_content=entry["q"] + " " + entry["a"],
            metadata={"department": dept, "audience": entry["audience"]}
        ))

# Chunk documents
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
split_docs = splitter.split_documents(docs)

# Build FAISS index
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"✅ FAISS vector store built!")
print(f"   Original documents : {len(docs)}")
print(f"   Chunks after split : {len(split_docs)}")
print(f"   Retriever top-k    : 2")

---
## 🧠 Step 6 — Initialise Gemini LLMs

We use two Gemini instances:
- **`router_llm`** — low temperature (0.2) for precise classification tasks (sentiment, routing)
- **`answer_llm`** — slightly higher temperature (0.3) for natural answer generation

In [ ]:
# LLM for classification tasks (routing + sentiment)
router_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)

# LLM for RAG answer generation
answer_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)

print("✅ Gemini LLMs initialised!")
print("   router_llm  → gemini-1.5-flash (temp=0.2) — for routing & sentiment")
print("   answer_llm  → gemini-1.5-flash (temp=0.3) — for RAG answer generation")

---
## 🗺️ Step 7 — Define Chat State

The `ChatState` TypedDict is the **shared memory** that flows between all nodes in the LangGraph workflow. Every node reads from it and writes back to it.

| Field | Type | Description |
|---|---|---|
| `message` | str | User's input query |
| `sentiment` | str | positive / neutral / negative |
| `department` | str | HR / Finance / Billing / Shipping / Unknown |
| `retrieved_docs` | List[Document] | FAISS retrieval results |
| `response` | str | Final assistant response |
| `escalate` | bool | Whether to route to human agent |

In [ ]:
class ChatState(TypedDict):
    message:        str
    sentiment:      str
    department:     str
    retrieved_docs: List[Document]
    response:       str
    escalate:       bool

print("✅ ChatState defined — 6 fields across the agent pipeline")

---
## 🤖 Step 8 — Define Agent Nodes

Each node is a pure Python function that takes a `ChatState`, does one thing, and returns an updated `ChatState`.

| Node | Role |
|---|---|
| `router_node` | Classify department from keywords |
| `sentiment_node` | Detect sentiment via Gemini LLM |
| `escalation_node` | Decide whether to escalate |
| `rag_node` | Retrieve docs + generate answer |
| `human_node` | Handle escalated queries |

In [ ]:
# ── 🚦 Router Node ─────────────────────────────────────────────────────────────
# Classifies the query into a department using keyword matching.
# Fast and reliable for well-defined department vocabularies.

def router_node(state: ChatState) -> ChatState:
    text = state["message"].lower()

    if any(w in text for w in ["leave", "pto", "holiday", "vacation", "performance",
                                "handbook", "harassment", "parental", "remote",
                                "payroll", "benefits", "training"]):
        state["department"] = "HR"

    elif any(w in text for w in ["expense", "reimbursement", "invoice", "fiscal",
                                  "per diem", "cost center", "vendor", "capital",
                                  "travel card", "purchase order"]):
        state["department"] = "Finance"

    elif any(w in text for w in ["card declined", "billing", "bill", "payment",
                                  "promo code", "refund", "bnpl", "installment",
                                  "vat", "charge", "download invoice"]):
        state["department"] = "Billing"

    elif any(w in text for w in ["shipping", "delivery", "track", "package",
                                  "order", "ship", "return", "expedited"]):
        state["department"] = "Shipping"

    else:
        state["department"] = "Unknown"

    return state

print("✅ router_node defined")

In [ ]:
# ── ❤️ Sentiment Node ──────────────────────────────────────────────────────────
# Uses Gemini LLM to classify sentiment as positive, neutral, or negative.
# Low temperature (0.2) ensures consistent, deterministic classification.

def sentiment_node(state: ChatState) -> ChatState:
    prompt = (
        "Classify the sentiment of the following message as exactly one word: "
        "positive, neutral, or negative.\n\n"
        f"Message: {state['message']}\n\n"
        "Respond with only the single word."
    )
    response = router_llm.invoke([HumanMessage(content=prompt)])
    state["sentiment"] = response.content.strip().lower()
    return state

print("✅ sentiment_node defined")

In [ ]:
# ── 🚨 Escalation Node ─────────────────────────────────────────────────────────
# Escalates if:
#   - sentiment is negative (frustrated/angry user needs human attention)
#   - department is Unknown (query out of scope for the knowledge base)

def escalation_node(state: ChatState) -> ChatState:
    if state["sentiment"] == "negative" or state["department"] == "Unknown":
        state["escalate"] = True
        state["response"] = "Escalated to a human agent. You'll be contacted shortly. 🙋"
    else:
        state["escalate"] = False
    return state

print("✅ escalation_node defined")

In [ ]:
# ── 📚 RAG Node ────────────────────────────────────────────────────────────────
# 1. Retrieves the top-2 most relevant documents from FAISS
# 2. Builds a grounded prompt with the retrieved context
# 3. Generates a natural language answer via Gemini

def rag_node(state: ChatState) -> ChatState:
    # Retrieve relevant documents
    docs = retriever.get_relevant_documents(state["message"])
    state["retrieved_docs"] = docs

    # Build grounded prompt
    context = "\n".join([d.page_content for d in docs])
    prompt = f"""You are ShopUNow's helpful AI assistant.
Use the context below to answer the user's question accurately and concisely.
If the answer isn't in the context, say: "I'm not certain — a human agent will assist you."

Context:
{context}

User Question: {state['message']}

Answer:"""

    response = answer_llm.invoke([HumanMessage(content=prompt)])
    state["response"] = response.content
    return state

print("✅ rag_node defined")

In [ ]:
# ── 🧑 Human Node ──────────────────────────────────────────────────────────────
# Terminal node for escalated queries.
# In production this would trigger a CRM ticket or support queue notification.

def human_node(state: ChatState) -> ChatState:
    state["response"] = (
        "Your query has been escalated to a human support agent. "
        "Our team will reach out to you shortly. 🙋"
    )
    return state

print("✅ human_node defined")

---
## 🔗 Step 9 — Build & Compile the LangGraph StateGraph

We wire all 5 nodes together into a directed graph:

```
[router] → [sentiment] → [escalation] → [human] → END
                                      ↘ [rag]   → END
```

The conditional edge from `escalation` routes based on `state["escalate"]`.

In [ ]:
# Build StateGraph
graph = StateGraph(ChatState)

# Register all nodes
graph.add_node("router",     router_node)
graph.add_node("sentiment",  sentiment_node)
graph.add_node("escalation", escalation_node)
graph.add_node("rag",        rag_node)
graph.add_node("human",      human_node)

# Define edges
graph.set_entry_point("router")
graph.add_edge("router",    "sentiment")
graph.add_edge("sentiment", "escalation")

# Conditional routing from escalation node
graph.add_conditional_edges(
    "escalation",
    lambda state: "human" if state["escalate"] else "rag",
    {"human": "human", "rag": "rag"}
)

graph.add_edge("human", END)
graph.add_edge("rag",   END)

# Compile the graph
app = graph.compile()

print("✅ LangGraph StateGraph compiled successfully!")
print("   Nodes     : router → sentiment → escalation → rag / human")
print("   Entry     : router")
print("   Exit      : END (via rag or human)")

---
## 🖼️ Step 10 — Visualise the Agent Workflow

LangGraph can export the StateGraph as a Mermaid diagram, which renders natively in notebooks.

In [ ]:
from IPython.display import display, Markdown

mermaid_code = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

---
## 🧪 Step 11 — Run Test Queries

We test 5 representative queries covering all departments and both escalation and RAG paths.

In [ ]:
test_queries = [
    "How do I apply for annual leave?",                        # HR  — RAG
    "I need help with my expense reimbursement.",              # Finance — RAG
    "How do I download my invoice?",                          # Billing — RAG
    "The delivery is late again, this is so frustrating!",    # Shipping — ESCALATE (negative)
    "thanks",                                                 # Unknown — ESCALATE
]

print("=" * 70)
print("ShopUNow AI Assistant — Test Run")
print("=" * 70)

for query in test_queries:
    initial_state = ChatState(
        message=query,
        sentiment="",
        department="",
        retrieved_docs=[],
        response="",
        escalate=False,
    )
    result = app.invoke(initial_state)

    escalated_label = "✅ ESCALATED" if result["escalate"] else "❌ NOT escalated"

    print(f"\n🧑 User     : {query}")
    print(f"   Dept     : {result['department']}")
    print(f"   Sentiment: {result['sentiment']}")
    print(f"   Status   : {escalated_label}")
    print(f"🤖 Assistant: {result['response']}")
    print("-" * 70)

---
## 💬 Step 12 — Interactive Assistant

Run the cell below for a live conversation with the ShopUNow Assistant.  
Type your query and press **Enter**. Type `quit` to stop.

> 💡 Try these to see different behaviours:
> - `"How do I apply for parental leave?"` → HR RAG answer
> - `"My package hasn't arrived and I'm furious!"` → Escalation
> - `"What's the refund timeline?"` → Billing RAG answer

In [ ]:
print("🛍️ ShopUNow Assistant — type 'quit' to stop.\n")
print("-" * 50)

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("\n🤖 Assistant: Goodbye! Have a great day 👋")
        break

    initial_state = ChatState(
        message=user_input,
        sentiment="",
        department="",
        retrieved_docs=[],
        response="",
        escalate=False,
    )
    result = app.invoke(initial_state)

    status = "🚨 ESCALATED" if result["escalate"] else f"📂 {result['department']}"
    print(f"🤖 Assistant [{status}]: {result['response']}")
    print("-" * 50)

---

## 🔮 What's Next?

This notebook demonstrates the core agentic workflow. The full project extends this with:

- **FastAPI REST endpoint** — production-ready `/ask` API with Pydantic validation
- **Contact detail collection** — escalated queries capture name, phone, email for human handoff
- **Modular codebase** — each node is a separate, testable Python module

Explore the full project: [github.com/suranjika/ShopUNow-AI-Assistant](https://github.com/suranjika/ShopUNow-AI-Assistant)

---

**Author:** Suranjika Sahu — Data & Analytics Leader | AI/GenAI | Jakarta  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-0077B5?style=flat&logo=linkedin)](https://linkedin.com/in/suranjika-sahu)  
[![GitHub](https://img.shields.io/badge/GitHub-ShopUNow-181717?style=flat&logo=github)](https://github.com/suranjika/ShopUNow-AI-Assistant)
